In [1]:
from pathlib import Path
import sys
import pandas as pd

In [2]:
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project_paths.py").exists()),
    None
)
if ROOT is None:
    raise RuntimeError("Racine du projet introuvable (project_paths.py non trouvé).")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [3]:
from project_paths import PROCESSED_DATA_DIR
path_of_df_nextBuy = PROCESSED_DATA_DIR / "nextbuy.pkl.gz"
df_nextBuy = df = pd.read_pickle(path_of_df_nextBuy , compression="gzip")

In [ ]:
#Ajout de features pouvant améliorer les prédictions du modèles déduite après eda.

#Combien de fois ce produit à été commander
df_nextBuy["user_product_count"] = (
    df_nextBuy
    .groupby(["user_id", "product_id"])["order_id"]
    .transform("nunique")
)

#frequence moyenne à laquelle le user fait des orders
df_nextBuy["user_recency"] = (
    df_nextBuy
    .groupby("user_id")["days_since_prior_order"]
    .transform("mean")
)

#popularité du produit en génerale

df_nextBuy["product_popularity"] = (
    df_nextBuy
    .groupby("product_id")["order_id"]
    .transform("count")
)


In [5]:
df_nextBuy.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department,user_product_count,user_recency,product_popularity
0,2,33120.0,1.0,1.0,202279.0,3.0,5.0,9.0,8.0,Organic Egg Whites,86.0,16.0,eggs,dairy eggs,3.0,15.648649,8129.0
1,2,28985.0,2.0,1.0,202279.0,3.0,5.0,9.0,8.0,Michigan Organic Kale,83.0,4.0,fresh vegetables,produce,3.0,15.648649,28486.0
2,2,9327.0,3.0,0.0,202279.0,3.0,5.0,9.0,8.0,Garlic Powder,104.0,13.0,spices seasonings,pantry,1.0,15.648649,2640.0
3,2,45918.0,4.0,1.0,202279.0,3.0,5.0,9.0,8.0,Coconut Butter,19.0,13.0,oils vinegars,pantry,3.0,15.648649,323.0
4,2,30035.0,5.0,0.0,202279.0,3.0,5.0,9.0,8.0,Natural Sweetener,17.0,13.0,baking ingredients,pantry,3.0,15.648649,222.0


In [5]:
output_path = PROCESSED_DATA_DIR / "nextbuy_enge.pkl.gz"
df_nextBuy.to_pickle(output_path, compression="gzip")